# Bronze Layer: Ingest Taxi Data

This notebook ingests raw taxi trip data from Parquet files into a Bronze Delta table.

## Imports

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

## Configuration

Define catalog, schema, and table paths for the bronze layer.

In [0]:
CATALOG = "chicago_city"
LANDING= f"/Volumes/{CATALOG}/raw/trips_raw"
SCHEMA="bronze"
BRONZE_TABLE="bronze_trips"
BRONZE_PATH=f"{CATALOG}.{SCHEMA}.{BRONZE_TABLE}"

## Load Raw Data

Read Parquet files from the landing zone and add ingestion timestamp.

In [0]:
father_df = spark.read.parquet(LANDING)
father_df = father_df.withColumn("ingested_at", F.current_timestamp())
father_df.printSchema()
print("Total rows in the dataset: ", father_df.count())

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: string (nullable = true)
 |-- trip_end_timestamp: string (nullable = true)
 |-- trip_seconds: double (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: double (nullable = true)
 |-- dropoff_census_tract: double (nullable = true)
 |-- pickup_community_area: double (nullable = true)
 |-- dropoff_community_area: double (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_centroid_latitude: double (nullable = true)
 |-- pickup_centroid_longitude: double (nullable = true)
 |-- pickup_centroid_location: string (nullable = true)
 |-- dropoff_centroid_latitude: double (nullable = true)
 |-- dropoff_centroid

## Add Partitioning Column

Create a year-month partition key from the trip start timestamp.

In [0]:
father_df = father_df.withColumn("year_month",
                                 F.date_format(
                                     F.col("trip_start_timestamp"),
                                     "yyyy-MM"
                                 ))

## Write to Bronze Table

Write data to the bronze Delta table with monthly partitioning.

In [0]:
father_df.write.format("delta").mode("append").partitionBy("year_month").option("mergeSchema", "true").saveAsTable(BRONZE_PATH)

## Optimize and Vacuum

Compact small files and clean up old file versions.

In [0]:
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")
# OPTIMIZE compacts small files (critical for partitioned Delta tables)
# VACUUM removes old file versions beyond the retention window
spark.sql(f"OPTIMIZE `{BRONZE_TABLE}`")
spark.sql(f"VACUUM `{BRONZE_TABLE}` RETAIN 168 HOURS")

DataFrame[path: string]

## Table History

View the Delta table's version history and operations.

In [0]:
spark.sql(f"DESCRIBE HISTORY `{BRONZE_TABLE}`").show(truncate=False)

+-------+-------------------+--------------+--------------------------+----------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------+------------+------------------------------------------------+
|version|timestamp          |userId        |userName                  |operation             |operationParameters                                                                                                                                                                                                                                        